# SEED-VII EEGNet x LoRA-LLM — 本地可编程 Pipeline

非交互式。所有配置通过环境变量或顶部变量设置。

## 0. 用户配置（无交互）

In [ ]:
import os, sys, shutil, time, re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

LOCAL_REPO_PATH      = os.environ.get('LOCAL_REPO_PATH', None)
GIT_CLONE_DIR        = Path(os.environ.get('GIT_CLONE_DIR', './EEG_OPUS'))
WORK_ROOT            = Path(os.environ.get('WORK_ROOT', './workspace'))
DATASET_ID           = os.environ.get('DATASET_ID', 'DEREKVERSE/SEED-VII')
LOCAL_DATASET_DIR    = WORK_ROOT / 'seedvii_ms_dataset'
NPZ_DIR              = WORK_ROOT / 'seedvii_npz'
RUN_DIR              = WORK_ROOT / 'runs' / 'run_valence3'
LLM_MODEL_DIR        = os.environ.get('LLM_MODEL_DIR', None)
LLM_MODEL_ID         = 'Qwen/Qwen2.5-0.5B-Instruct'
LLM_CACHE_DIR        = WORK_ROOT / 'models'
TRAIN_BATCH_SIZE     = int(os.environ.get('TRAIN_BATCH_SIZE', '96'))
TRAIN_EPOCHS         = int(os.environ.get('TRAIN_EPOCHS', '50'))
TRAIN_DEVICE         = os.environ.get('TRAIN_DEVICE', 'auto')
TRAIN_RESUME         = os.environ.get('TRAIN_RESUME', 'true').lower() == 'true'
MODELSCOPE_TOKEN     = os.environ.get('MODELSCOPE_TOKEN', None)
MAX_WORKERS          = int(os.environ.get('MAX_WORKERS', '8'))

for d in [WORK_ROOT, LOCAL_DATASET_DIR, NPZ_DIR, RUN_DIR, LLM_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('[CONFIG] OK')

## 1. 环境

In [ ]:
if LOCAL_REPO_PATH:
    REPO = Path(LOCAL_REPO_PATH); assert REPO.exists()
else:
    REPO = Path(GIT_CLONE_DIR).resolve()
    if not (REPO/'seedvii_modal_contrastive_lora'/'pyproject.toml').exists():
        !git clone --depth 1 https://github.com/PRIMOCOSMOS/EEG_OPUS.git {REPO} 2>&1 | tail -1
PROJ = REPO / 'seedvii_modal_contrastive_lora'

%pip install -q -r {PROJ/'requirements.txt'} 2>&1 | tail -1
%pip install -q modelscope 2>&1 | tail -1
%pip install -q -e {PROJ} 2>&1 | tail -1
if str(PROJ) not in sys.path: sys.path.insert(0, str(PROJ))
print('[OK]')

## 2. 数据 — 没有就去 ModelScope 下载

In [ ]:
from modelscope.hub.api import HubApi
from modelscope.hub.file_download import dataset_file_download
from seedvii_contrastive.scripts.download_modelscope_seedvii import find_downloaded_paths
from seedvii_contrastive.data.discovery import SUBJECT_FILE_NAMES

lock = threading.Lock()

def _mat_count(d):
    if not d or not Path(d).exists(): return 0
    return len([f for f in Path(d).glob('*.mat') if f.name in SUBJECT_FILE_NAMES])

du = shutil.disk_usage(WORK_ROOT)
print(f'[DISK] {WORK_ROOT}: {du.free/(1024**3):.1f} GB free')

eeg_root_test, csv_test = find_downloaded_paths(LOCAL_DATASET_DIR)
if eeg_root_test and _mat_count(eeg_root_test) >= 20 and csv_test:
    print(f'[SKIP] {_mat_count(eeg_root_test)} .mat already local')
else:
    print(f'[LIST] {DATASET_ID} ...')
    api = HubApi(token=MODELSCOPE_TOKEN)
    all_f, page = [], 1
    while True:
        batch = api.get_dataset_files(repo_id=DATASET_ID, revision='master', root_path='/', recursive=True, page_number=page, page_size=200)
        if not batch: break
        all_f.extend(batch); page += 1
        if len(batch) < 200: break
    
    targets, sizes = {}, {}
    for f in all_f:
        path = f.get('Path') or f.get('Name') or ''
        name = Path(path).name
        if re.match(r'^([1-2][0-9]|[1-9])\.mat$', name) or name == 'text_protocol.csv':
            targets[name] = path; sizes[name] = int(f.get('Size', 0))
    print(f'[FIND] {len(targets)} target files')
    for k in sorted(targets): print(f'  {k:25s} ({sizes[k]/(1024**2):.0f} MB)')
    
    missing = [f'{i}.mat' for i in range(1,21) if f'{i}.mat' not in targets]
    if missing: raise RuntimeError(f'Not found: {missing}')
    
    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    to_dl = []
    for name, rp in sorted(targets.items()):
        dest = LOCAL_DATASET_DIR / name
        min_sz = int(sizes.get(name, 0) * 0.9)
        if dest.exists() and dest.stat().st_size >= min_sz: continue
        dest.unlink(missing_ok=True)
        to_dl.append((name, rp))
    
    if to_dl:
        print(f'[DL] {len(to_dl)} files, {MAX_WORKERS} workers ...')
        t0 = time.time(); failures = []
        def _dl_one(name, rp, dest_dir, exp_sz, max_tries=5):
            dest = Path(dest_dir)/name; min_b = int(exp_sz*0.9)
            for attempt in range(1, max_tries+1):
                try:
                    dl = Path(dataset_file_download(dataset_id=DATASET_ID, file_path=rp, revision='master', local_dir=str(dest_dir), token=MODELSCOPE_TOKEN))
                    if dl != dest: dest.unlink(missing_ok=True); shutil.copy2(str(dl), str(dest)); dl.unlink(missing_ok=True)
                    if dest.stat().st_size < min_b: dest.unlink(missing_ok=True); raise RuntimeError('Size')
                    with lock: print(f'  [OK] {name:12s} ({dest.stat().st_size/(1024**2):7.1f} MB)')
                    return (name, dest, None)
                except Exception as e:
                    dest.unlink(missing_ok=True)
                    if attempt < max_tries:
                        w = min(2**attempt, 60)
                        with lock: print(f'  [RETRY {attempt}/{max_tries}] {name}: {e} - {w}s')
                        time.sleep(w)
                    else:
                        with lock: print(f'  [FAIL] {name}: {e}')
                        return (name, None, str(e))
            return (name, None, 'unknown')
        with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(to_dl))) as pool:
            futs = {pool.submit(_dl_one, n, r, LOCAL_DATASET_DIR, sizes[n]): n for n, r in to_dl}
            for fut in as_completed(futs):
                n, _, err = fut.result()
                if err: failures.append(n)
        elapsed = time.time()-t0
        ok = len(to_dl)-len(failures)
        print(f'[DL] {ok}/{len(to_dl)} OK ({elapsed:.0f}s, {elapsed/max(1,ok):.1f}s/file)')
        if failures: raise RuntimeError(f'Failed: {failures}')

EEG_ROOT, TEXT_CSV = find_downloaded_paths(LOCAL_DATASET_DIR)
assert EEG_ROOT and TEXT_CSV
print(f'[DATA] EEG={EEG_ROOT} CSV={TEXT_CSV} mats={_mat_count(EEG_ROOT)}')

## 3. LLM + NPZ + 训练

In [ ]:
resolved = None
if LLM_MODEL_DIR and (Path(LLM_MODEL_DIR)/'config.json').exists(): resolved = Path(LLM_MODEL_DIR)
if resolved is None:
    for c in sorted(LLM_CACHE_DIR.rglob('config.json'), key=lambda x: len(str(x))):
        if 'qwen' in str(c).lower(): resolved = c.parent; break
if resolved is None:
    from modelscope import snapshot_download
    resolved = Path(snapshot_download(LLM_MODEL_ID, cache_dir=str(LLM_CACHE_DIR)))
print(f'[LLM] {resolved}')

if not (NPZ_DIR/'index.csv').exists():
    !python -m seedvii_contrastive.scripts.preprocess_npz --input-root {EEG_ROOT} --output-dir {NPZ_DIR} --subjects 1-20 --window-sec 4 --stride-sec 4 --center-ratio 0.60 --max-windows-per-clip 12 --shard-size 512

import yaml
cfg = yaml.safe_load(open(PROJ/'configs'/'modelscope_default.yaml'))
cfg['data'].update({k: str(v) for k, v in dict(modelscope_dataset_id=DATASET_ID, local_dataset_dir=str(LOCAL_DATASET_DIR), eeg_root=str(EEG_ROOT), text_csv_path=str(TEXT_CSV), npz_dir=str(NPZ_DIR)).items()})
cfg['runtime']['output_dir'] = str(RUN_DIR)
cfg['runtime']['device'] = TRAIN_DEVICE
cfg['model']['llm']['model_name_or_path'] = str(resolved)
cfg['train'].update(dict(batch_size=TRAIN_BATCH_SIZE, epochs=TRAIN_EPOCHS, resume=TRAIN_RESUME))
rc = RUN_DIR/'config.yaml'; RUN_DIR.mkdir(parents=True, exist_ok=True)
yaml.safe_dump(cfg, open(rc,'w'), allow_unicode=True, sort_keys=False)
!python -m seedvii_contrastive.scripts.train_contrastive --config {rc}

In [ ]:
BEST = RUN_DIR/'best.pt'
if BEST.exists():
    !python -m seedvii_contrastive.scripts.encode_eeg --config {rc} --checkpoint {BEST} --split val --out {RUN_DIR/'emb.npz'}
    import numpy as np, matplotlib.pyplot as plt
    from sklearn.manifold import TSNE
    d = np.load(RUN_DIR/'emb.npz')
    e = TSNE(2, random_state=42, perplexity=30).fit_transform(d['embedding'])
    fig, ax = plt.subplots(figsize=(8,6))
    for c, clr, nm in zip(range(3), ['#e74c3c','#95a5a6','#2ecc71'], ['Neg','Neu','Pos']):
        ax.scatter(e[d['label']==c,0], e[d['label']==c,1], c=clr, label=nm, alpha=.5, s=8)
    ax.legend(); ax.set_title('t-SNE'); plt.tight_layout()
    plt.savefig(RUN_DIR/'tsne.png', dpi=150); plt.show()
else:
    print('[WARN] No best.pt')